In [4]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-pinecone \
    langchain-groq \
    pinecone \
    sentence-transformers \
    python-dotenv \
    pypdf \
    langgraph \
    langgraph-prebuilt \
    requests==2.32.4


In [5]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
from google.colab import files

uploaded = files.upload()

Saving Medical_book.pdf to Medical_book (1).pdf


In [6]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_file(filename):
    loader = PyPDFLoader(filename)
    documents = loader.load()
    return documents

In [12]:
import os

print(os.path.exists("Medical_book.pdf"))



True


In [13]:
extracted_data = load_pdf_file("Medical_book.pdf")

In [14]:
#Split the Data into Text Chunks- fixed chunking and chunk overlapping
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [19]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))


Length of Text Chunks 5860


In [21]:
from langchain_huggingface import HuggingFaceEmbeddings


In [22]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


In [23]:
embeddings = download_hugging_face_embeddings()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
query_result = embeddings.embed_query("My name is  yashi agrawal")
print(query_result)

[-0.06319135427474976, 0.02372761256992817, 0.048564087599515915, -0.004100365564227104, -0.07151380181312561, -0.058539263904094696, 0.15603944659233093, -0.03164749592542648, 0.06442633271217346, -0.018506363034248352, 0.030144119635224342, -0.1392727643251419, 0.0057672737166285515, -0.03289811313152313, 0.013060848228633404, 0.046677365899086, 0.07043413817882538, 0.06856980174779892, -0.02283799648284912, -0.10005834698677063, -0.08741208910942078, 0.08710835874080658, -0.030015068128705025, -0.05795977637171745, -0.029137561097741127, 0.03800029680132866, 0.037174370139837265, 0.09267019480466843, 0.007630688138306141, -0.03174344822764397, -0.04487058147788048, 0.025265773758292198, 0.03152383863925934, 0.034637317061424255, -0.03618764877319336, 0.002887796377763152, -0.044552579522132874, 0.029890339821577072, 0.028999872505664825, -0.0028610455337911844, -0.011211785487830639, -0.02202301099896431, 0.047302354127168655, -0.04470236971974373, -0.013735177926719189, -0.02745840

In [28]:
import os
os.environ["GROQ_API_KEY"] ="...."
os.environ["PINECONE_API_KEY"] ="....."

In [29]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

In [30]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "medical-chatbot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully.")
else:
    print("Index already exists. Using existing index.")

Index created successfully.


In [31]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
# PineconeVectorStore: Connects LangChain with Pinecone to store and search document embeddings.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [32]:
docsearch

In [36]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":5})

In [39]:
retrieved_docs = retriever.invoke("What is food ?")

In [40]:
retrieved_docs

[Document(id='f124b34f-f97d-4ec5-a946-a6d4e6f2baa4', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 426.0, 'page_label': '427', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Medical_book.pdf', 'total_pages': 637.0}, page_content='• EEG (electroencephalograph) biofeedback . By mea-\nsuring brainwave activity and teaching the ADHD\npatient which type of brainwave is associated with\nattention, EEG biofeedback attempts to train patients to\ngenerate the desired brainwave activity.\n• Dietary therapy. Based in part on the Feingold food\nallergy diet, dietary therapy focuses on a nutritional\nplan that is high in protein and complex carbohydrates\nand free of white sugar and salicylate-containing foods'),
 Document(id='5de82eca-ab8e-4adc-a301-9e67c337a306', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 484.0, 'page_label': '485', 'prod

In [41]:
from langchain_groq import ChatGroq
llm = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0
)

In [48]:
# ChatPromptTemplate: Creates a structured prompt template for sending
# system instructions and user messages to the LLM.
from langchain_core.prompts import ChatPromptTemplate
# RunnablePassthrough: Passes the input data through a LangChain chain
# without changing it.
from langchain_core.runnables import RunnablePassthrough
# StrOutputParser: Converts the LLM's response into a simple string.
from langchain_core.output_parsers import StrOutputParser


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Donot respond to any question outside the context provided. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "Context:\n{context}"
)

# Creates a chat prompt containing a system message and a human/user message.
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [49]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [52]:
response = rag_chain.invoke("How manhy days in  month?")
print(response)

I’m sorry, but I don’t know.
